# Stable Log-Softmax and Cross-Entropy From Logits (Teaching Notes)

## 1) Problem Setup

Given one sample:

- Logits: $\mathbf{z} \in \mathbb{R}^{C}$
- True class: $y \in \{0, 1, \dots, C-1\}$

We want cross-entropy from logits:

$$
\mathcal{L} = -\log p_y,
\quad
p_i = \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}}
$$

For a batch of size $N$ (logits shape $(N, C)$):

$$
\mathcal{L}_{\text{mean}} = \frac{1}{N}\sum_{n=1}^{N} -\log p_{n,y_n}
$$

## 2) Why Naive Softmax -> Log Is Risky

Naive approach:

1. Compute $p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$
2. Compute $-\log p_y$

Numerical issue:

- If some $z_i$ is very large, $e^{z_i}$ can overflow to $\infty$.
- This can create `inf`, `nan`, and unstable training.

## 3) Stable Reformulation

Use:

$$
\log p_i
= \log\left(\frac{e^{z_i}}{\sum_j e^{z_j}}\right)
= z_i - \log\sum_j e^{z_j}
$$

So,

$$
\log\text{-softmax}(z_i) = z_i - \log\sum_j e^{z_j}
$$

And cross-entropy for class $y$ is:

$$
\mathcal{L} = -\log p_y = \log\sum_j e^{z_j} - z_y
$$

This is the key interview identity:

$$
\boxed{\mathcal{L} = \log\sum_j e^{z_j} - z_y}
$$

## 4) Log-Sum-Exp Trick

Compute $\log\sum_j e^{z_j}$ safely as:

$$
\log\sum_j e^{z_j}
= m + \log\sum_j e^{z_j - m},
\quad m = \max_j z_j
$$

Why safe:

- $z_j - m \le 0$
- $e^{z_j-m} \le 1$
- No overflow in exponentials

In PyTorch, `torch.logsumexp` already implements this stably.

## 5) Batch Formula (What We Implement)

For each sample $n$:

$$
\mathcal{L}_n = \log\sum_j e^{z_{n,j}} - z_{n,y_n}
$$

Then reduce with `mean`, `sum`, or `none`.

## 6) Math -> Code Mapping

- $\log\sum_j e^{z_{n,j}}$ -> `torch.logsumexp(logits, dim=-1)`
- $z_{n,y_n}$ -> `logits[torch.arange(N), targets]`
- per-sample loss -> `lse - z_y`

## 7) Ignore Index Behavior

When `ignore_index` is used:

1. Build mask: `valid = targets != ignore_index`
2. Compute per-sample losses
3. Drop ignored elements **before** reduction

This matches `torch.nn.functional.cross_entropy` semantics.

## 8) Common Interview Pitfalls

1. `torch.log(torch.softmax(logits, dim=-1))` (less stable than `logsumexp` route)
2. Wrong indexing like `logits[:, targets]`
3. Applying ignore-mask after reduction
4. Forgetting to test all reductions (`mean`, `sum`, `none`)

## 9) Gradient Insight

For one sample:

$$
\frac{\partial \mathcal{L}}{\partial z_i} = p_i - \mathbf{1}_{i=y}
$$

This is why softmax + cross-entropy is efficient and widely used.

## 10) Interview One-Liner

Cross-entropy from logits is `logsumexp(logits) - target_logit`; `logsumexp` gives numerical stability by max-shifting inside the exponentials.

In [3]:
!pip3 install torch torchvision

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached pillow-12.1.1-cp311-cp311-win_amd64.whl.metadata (9.0 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/113.7 MB ? eta -:--:--
   - -------------------------------------- 4.2/113.7 MB 25.0 MB/s eta 0:00:05
   ---- ----------------------------------- 13.4/113.7 MB 35.0 MB/s eta 0:00:03
   -------- ------------------------------- 24.6/113.7 MB 41.0 MB/s eta 0:00:03
   ------------ --------------------------- 36.2/113.7 MB 44.2 MB/s eta 0:00:02
   ---------------- ----------------------- 47.2/113.7 MB 46.2 MB/s eta 0:00:02
   -------------------- ------------------- 59.0/113.7 MB 47.5 MB/s eta 0:00:02
   ------------------------ --------------- 70.5/113.7 MB 48.3 MB/s eta 0:00:01
   ---------------------------- ----------- 

In [4]:
import torch
import torch.nn.functional as F


def log_softmax(logits, dim=-1):
    """Numerically stable log-softmax via logsumexp."""
    return logits - torch.logsumexp(logits, dim=dim, keepdim=True)


def cross_entropy_from_logits_math(logits, targets, reduction="mean"):
    """
    Teaching version directly from:
        L_n = logsumexp(z_n) - z_{n, y_n}

    logits: (N, C)
    targets: (N,)
    reduction: "mean" | "sum" | "none"
    """
    if logits.ndim != 2:
        raise ValueError(f"Expected logits shape (N, C), got {tuple(logits.shape)}")
    if targets.ndim != 1:
        raise ValueError(f"Expected targets shape (N,), got {tuple(targets.shape)}")
    if logits.shape[0] != targets.shape[0]:
        raise ValueError("Batch size mismatch between logits and targets")
    if reduction not in {"mean", "sum", "none"}:
        raise ValueError(f"Unsupported reduction: {reduction}")

    n = logits.shape[0]
    rows = torch.arange(n, device=logits.device)

    lse = torch.logsumexp(logits, dim=-1)   # (N,)
    z_y = logits[rows, targets]             # (N,)
    loss = lse - z_y                        # (N,)

    if reduction == "none":
        return loss
    if reduction == "sum":
        return loss.sum()
    return loss.mean()


def cross_entropy_from_logits(logits, targets, ignore_index=None, reduction="mean"):
    """
    Full version with optional ignore_index.
    Matches torch.nn.functional.cross_entropy behavior for class-index targets.

    logits: (N, C)
    targets: (N,)
    """
    if logits.ndim != 2:
        raise ValueError(f"Expected logits shape (N, C), got {tuple(logits.shape)}")
    if targets.ndim != 1:
        raise ValueError(f"Expected targets shape (N,), got {tuple(targets.shape)}")
    if logits.shape[0] != targets.shape[0]:
        raise ValueError("Batch size mismatch between logits and targets")
    if reduction not in {"mean", "sum", "none"}:
        raise ValueError(f"Unsupported reduction: {reduction}")

    n = logits.shape[0]
    rows = torch.arange(n, device=logits.device)

    if ignore_index is not None:
        valid = targets != ignore_index
        safe_targets = targets.clone()
        safe_targets[~valid] = 0  # placeholder index for ignored rows
    else:
        valid = torch.ones_like(targets, dtype=torch.bool)
        safe_targets = targets

    lse = torch.logsumexp(logits, dim=-1)      # (N,)
    z_y = logits[rows, safe_targets]           # (N,)
    per_sample = lse - z_y                     # (N,)

    if reduction == "none":
        out = torch.zeros_like(per_sample)
        out[valid] = per_sample[valid]
        return out

    kept = per_sample[valid]
    if reduction == "sum":
        return kept.sum()

    # Same edge behavior as PyTorch: mean over non-ignored elements.
    if kept.numel() == 0:
        return torch.tensor(float("nan"), device=logits.device, dtype=logits.dtype)
    return kept.mean()


def test_log_softmax_matches_torch():
    torch.manual_seed(0)
    logits = torch.randn(8, 4) * 1000.0

    my_val = log_softmax(logits, dim=-1)
    torch_val = F.log_softmax(logits, dim=-1)

    assert torch.allclose(my_val, torch_val, atol=1e-6)


def test_ce_math_form_matches_torch():
    torch.manual_seed(1)
    logits = torch.randn(10, 5)
    targets = torch.randint(0, 5, (10,))

    my_mean = cross_entropy_from_logits_math(logits, targets, reduction="mean")
    torch_mean = F.cross_entropy(logits, targets, reduction="mean")

    my_sum = cross_entropy_from_logits_math(logits, targets, reduction="sum")
    torch_sum = F.cross_entropy(logits, targets, reduction="sum")

    my_none = cross_entropy_from_logits_math(logits, targets, reduction="none")
    torch_none = F.cross_entropy(logits, targets, reduction="none")

    assert torch.allclose(my_mean, torch_mean, atol=1e-6)
    assert torch.allclose(my_sum, torch_sum, atol=1e-6)
    assert torch.allclose(my_none, torch_none, atol=1e-6)


def test_ce_with_ignore_index_matches_torch():
    torch.manual_seed(2)
    logits = torch.randn(12, 7)
    targets = torch.randint(-1, 7, (12,))
    ignore_index = -1

    my_mean = cross_entropy_from_logits(
        logits, targets, ignore_index=ignore_index, reduction="mean"
    )
    torch_mean = F.cross_entropy(
        logits, targets, ignore_index=ignore_index, reduction="mean"
    )

    my_sum = cross_entropy_from_logits(
        logits, targets, ignore_index=ignore_index, reduction="sum"
    )
    torch_sum = F.cross_entropy(
        logits, targets, ignore_index=ignore_index, reduction="sum"
    )

    my_none = cross_entropy_from_logits(
        logits, targets, ignore_index=ignore_index, reduction="none"
    )
    torch_none = F.cross_entropy(
        logits, targets, ignore_index=ignore_index, reduction="none"
    )

    assert torch.allclose(my_mean, torch_mean, atol=1e-6)
    assert torch.allclose(my_sum, torch_sum, atol=1e-6)
    assert torch.allclose(my_none, torch_none, atol=1e-6)


def run_all_tests():
    test_log_softmax_matches_torch()
    test_ce_math_form_matches_torch()
    test_ce_with_ignore_index_matches_torch()
    print("All tests passed.")


run_all_tests()

All tests passed.
